# NEXUS: OpenMP, MPI and CUDA runs on Colab

Runs the three toolchain-dependent parts of NEXUS that cannot run on the Windows development
machine: the OpenMP benchmark, the MPI benchmark (2 ranks) and the CUDA demo on the Colab GPU,
plus the complete test suite (including the Verilog co-simulation).

**How to run:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
The last cell downloads `nexus_colab_results.zip`; copy its contents into `results/colab/` of the
repository.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nvcc --version | tail -2

In [ ]:
# Toolchains: Ninja, Flex/Bison, OpenMPI, Icarus Verilog, Yosys (gcc with OpenMP and CUDA are preinstalled)
!apt-get -qq update > /dev/null && apt-get -qq install -y ninja-build flex bison libopenmpi-dev openmpi-bin iverilog yosys > /dev/null
!cmake --version | head -1; mpicxx --version | head -1; iverilog -V 2>&1 | head -1; yosys -V

In [ ]:
import os
# OpenMPI refuses to run as root (Colab runs as root) unless these are set; the benchmark uses --oversubscribe.
os.environ["OMPI_ALLOW_RUN_AS_ROOT"] = "1"
os.environ["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"
!rm -rf nexus-platform && git clone -q https://github.com/gdtsitlauri/nexus-platform.git
%cd nexus-platform
!git log -1 --format='commit %h  %ad' --date=short

In [ ]:
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DNEXUS_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 2>&1 | grep -Ei "openmp|mpi|cuda|error" ; echo configured
!cmake --build build 2>&1 | tail -2
!ls build/bin | grep -E "openmp-bench|mpi-bench|gpu-bench|simd-bench"

In [ ]:
!mkdir -p results/colab
# Full suite: with gcc/OpenMP, OpenMPI and CUDA present the openmp, mpi, bench-wrapper and gpu tests are registered and run.
!cd build && ctest --output-on-failure -j2 2>&1 | tee ../results/colab/ctest.log | tail -15

In [ ]:
# The three benchmarks directly
!./build/bin/openmp-bench --kernel all --size 1024 --threads 2 | tee results/colab/openmp.txt
!mpiexec --oversubscribe -n 2 ./build/bin/mpi-bench --kernel all --size 1024 | tee results/colab/mpi.txt
!./build/bin/gpu-bench | tee results/colab/gpu.txt
!python3 scripts/parallel_bench.py --all --gpu --build-dir build --repo-root . --csv results/colab/parallel_bench.csv --markdown results/colab/parallel_bench.md | tail -20

In [ ]:
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader > results/colab/environment.txt
!(nvcc --version | tail -1; gcc --version | head -1; mpicxx --version | head -1; cmake --version | head -1) >> results/colab/environment.txt
!cd results && zip -qr ../nexus_colab_results.zip colab && cd .. && ls -la nexus_colab_results.zip
from google.colab import files
files.download("nexus_colab_results.zip")